## Data Import

This file includes an additional data preparation and feature engineering for the predictive analytics in task 4.

In [2]:
import pandas as pd
from datetime import timedelta
import holidays

Load the .csv files that were created in the data preparation.

In [30]:
cd = pd.read_csv("../00_Productive_Data/charging_data_clean.csv", parse_dates=["disconnectTime", "connectionTime", "doneChargingTime"])
wd = pd.read_csv("../00_Productive_Data/weather_data_clean.csv", parse_dates=["timestamp"])

# Drop the index column - not needed
wd.drop(columns={"Unnamed: 0"}, inplace=True)

Make sure that the datetime timestamps are in the right format

In [31]:
wd["timestamp"] = pd.to_datetime(wd["timestamp"], utc=True)
wd["timestamp"] = wd["timestamp"].dt.round("h")
wd["timestamp"] = wd["timestamp"].dt.tz_convert("America/Los_Angeles")
wd.rename(columns={"timestamp": "datetime"}, inplace=True)

Split the data for site 1 and site 2.

In [32]:
cd1 = cd[cd["siteID"] == 1]
cd2 = cd[cd["siteID"] == 2]

print("There are " + str(len(cd)) + " overall charging sessions.")
print("There are " + str(len(cd1)) + " charging sessions at site 1.")
print("There are " + str(len(cd2)) + " charging sessions at site 2.")

There are 60922 overall charging sessions.
There are 31590 charging sessions at site 1.
There are 29332 charging sessions at site 2.


For each site, count the existing parking spots. Therefore, we assume that every existing parking spot was at least usedonce in the dataset).

In [33]:
num_spots_cd1 = cd1['spaceID'].nunique()
num_spots_cd2 = cd2['spaceID'].nunique()

print("There are " + str(num_spots_cd1) + " existing parking spaces at site 1.")
print("There are " + str(num_spots_cd2) + " existing parking spaces at site 2.")

There are 52 existing parking spaces at site 1.
There are 54 existing parking spaces at site 2.


## Data Aggregation

The target metric we aim to predict in this task is the **hourly utilization**. Therefore we need to transform the data in a way, so that it reflects our target metric.
That is why we need to **aggregate the individual sessions** into hourly 'buckets' (i.e. time intervals), before we start predicting the hourly utilization. In order to aggregate the sessions hourly, we first need to cut/disaggregate sessions so that they fit into our hourly buckets, since many sessions stretch across several of those intervals.

Here is the function that disaggregates the sessions into hourly pieces:

In [ ]:
def disaggregate_session(row):
    """
    This function disaggregates a charging session into its hourly parts by splitting it at each full hour. 
    The resulting parts of the session are stored in a list and returned at the end.
    """
    current = row['connectionTime']
    end = row['disconnectTime']
    rows = [] # Empty list of returned rows
    while current < end:
        # Ensure that intervals stay within a session's bounds
        if current == current.ceil("h"):
            next_hour = min(current + timedelta(hours=1), end)
        else:
            next_hour = min(current.ceil("h"), end)
        rows.append({'id': row['id'], 
                     'connectionTime': row['connectionTime'], 
                     'disconnectTime': row['disconnectTime'],
                     'inHourStartTime': current,
                     'inHourEndTime': next_hour,
                     'minutesInHour' : ((next_hour - current).seconds) / 60})
        current = next_hour
    return rows

And here is the function that aggregates them:

In [44]:
def agg_rows(p_df):
    """
    This function aggregates disaggregated charging sessions into hourly buckets. 
    Each row in the returned df represents an hourly bucket. \n
    The key metric that is added through this function is the 'total minutes of parking' that are added from all subsets in an hourly bucket. \n
    Parameters:
    - 'p_df': Series | Series object with the disaggregated charging sessions
    """
    # Generate a range of hourly timestamps for the entire year
    start_time = f'2018-04-24 00:00:00'
    end_time = f'2021-09-14 23:00:00'
    hourly_range = pd.date_range(start=start_time, end=end_time, freq='h', tz="America/Los_Angeles")

    # Create a DataFrame with the timestamps
    this_df = pd.DataFrame(hourly_range, columns=['datetime'], index=hourly_range)

    # Add additional columns if needed (e.g., placeholder values)
    this_df['hour'] = this_df['datetime'].map(lambda x: x.hour)
    this_df['weekday'] = this_df['datetime'].map(lambda x: x.weekday())
    this_df['dayOfMonth'] = this_df['datetime'].map(lambda x: x.day)
    this_df['month'] = this_df['datetime'].map(lambda x: x.month)
    this_df['year'] = this_df['datetime'].map(lambda x: x.year)
    this_df['total minutes of parking'] = 0.0

    # Fill dataframe
    for session in p_df:
        for cur_row in session:
            # Optimize method by splitting data set that is searched
            cur_connectionTime = cur_row["inHourStartTime"]
            cur_minutesInHour = cur_row["minutesInHour"]
            cur_floor_connectionTime = cur_connectionTime.floor('h')

            this_df.loc[cur_floor_connectionTime, "total minutes of parking"] += cur_minutesInHour
    
    return this_df

We now apply these functions to create a new dataframe:

In [21]:
# Disaggregate
dis_cd1 = cd1.apply(disaggregate_session, axis=1)
dis_cd2 = cd2.apply(disaggregate_session, axis=1)

# Exemplary disaggregated session
dis_cd1[0]

[{'id': '5c36631ef9af8b4639a8e5bc',
  'connectionTime': Timestamp('2018-10-08 15:44:47-0700', tz='UTC-07:00'),
  'disconnectTime': Timestamp('2018-10-08 17:59:14-0700', tz='UTC-07:00'),
  'inHourStartTime': Timestamp('2018-10-08 15:44:47-0700', tz='UTC-07:00'),
  'inHourEndTime': Timestamp('2018-10-08 16:00:00-0700', tz='UTC-07:00'),
  'minutesInHour': 15.216666666666667},
 {'id': '5c36631ef9af8b4639a8e5bc',
  'connectionTime': Timestamp('2018-10-08 15:44:47-0700', tz='UTC-07:00'),
  'disconnectTime': Timestamp('2018-10-08 17:59:14-0700', tz='UTC-07:00'),
  'inHourStartTime': Timestamp('2018-10-08 16:00:00-0700', tz='UTC-07:00'),
  'inHourEndTime': Timestamp('2018-10-08 17:00:00-0700', tz='UTC-07:00'),
  'minutesInHour': 60.0},
 {'id': '5c36631ef9af8b4639a8e5bc',
  'connectionTime': Timestamp('2018-10-08 15:44:47-0700', tz='UTC-07:00'),
  'disconnectTime': Timestamp('2018-10-08 17:59:14-0700', tz='UTC-07:00'),
  'inHourStartTime': Timestamp('2018-10-08 17:00:00-0700', tz='UTC-07:00'),


In [22]:
# Aggregate
agg_cd1 = agg_rows(dis_cd1, 2018, 2021)
agg_cd2 = agg_rows(dis_cd2, 2018, 2021)

# Structure of the newly aggregated dataframe
agg_cd1.info()

<class 'pandas.core.frame.DataFrame'>
DatetimeIndex: 29760 entries, 2018-04-24 00:00:00-07:00 to 2021-09-14 23:00:00-07:00
Freq: h
Data columns (total 7 columns):
 #   Column                    Non-Null Count  Dtype                              
---  ------                    --------------  -----                              
 0   datetime                  29760 non-null  datetime64[ns, America/Los_Angeles]
 1   hour                      29760 non-null  int64                              
 2   weekday                   29760 non-null  int64                              
 3   dayOfMonth                29760 non-null  int64                              
 4   month                     29760 non-null  int64                              
 5   year                      29760 non-null  int64                              
 6   total minutes of parking  29760 non-null  float64                            
dtypes: datetime64[ns, America/Los_Angeles](1), float64(1), int64(5)
memory usage: 2.8 MB


Now that we have the charging data in an hourly format, we can merge the hourly weather data. To ensure temporal consistency, we apply an inner join on the hourly timestamp datetime, that only hours that exist in both datasets are retained and hours without charging activity without weather observations are removed:

In [23]:
agg_cd1["datetime"] = pd.to_datetime(agg_cd1["datetime"])
agg_cd2["datetime"] = pd.to_datetime(agg_cd2["datetime"])
wd["datetime"] = pd.to_datetime(wd["datetime"])

merged_cd1 = pd.merge(
    agg_cd1,
    wd,
    how="inner",
    on="datetime"
)

merged_cd2 = pd.merge(
    agg_cd2,
    wd,
    how="inner",
    on="datetime"
)

print("Merged CD1 shape:", merged_cd1.shape)
print("Merged CD2 shape:", merged_cd2.shape)
print("Missing values CD1:", merged_cd1.isna().sum().sum())

Merged CD1 shape: (23594, 14)
Merged CD2 shape: (23594, 14)
Missing values CD1: 0


## Feature Engineering

Since holidays can strongly influence demand patterns, we add them to the dataframe.

In [25]:
us_holidays = holidays.US(years=[2018, 2019, 2020, 2021])
merged_cd1['is_holiday'] = merged_cd1["datetime"].apply(lambda x: 1 if x in us_holidays else 0)
merged_cd2['is_holiday'] = (merged_cd2['datetime']).dt.date.isin(us_holidays).astype(int)

Since also the Corona-Pandemic can strongly influence demand patterns, we add them to the dataframe.

In [26]:
merged_cd1["covid"] = ((merged_cd1["year"] == 2020) & (merged_cd1["month"] >= 3)).astype(int)
merged_cd2["covid"] = ((merged_cd2["year"] == 2020) & (merged_cd2["month"] >= 3)).astype(int)

We now drop all columns, that we cannot use for our neural network:
- "datetime": We have encoded that as several distinct columns for hour, day, month etc.
- "cloud_cover_description": No real information loss, since it is only a mapping of cloud_cover

In [27]:
merged_cd1 = merged_cd1.drop(columns=["datetime", "cloud_cover_description"])
merged_cd2 = merged_cd2.drop(columns=["datetime", "cloud_cover_description"])

In [28]:
merged_cd1.to_csv("FINAL1.csv")
merged_cd1.to_csv("FINAL2.csv")

Since the **user_inputs** are only available for a minor subset of our data, we refrain from using it for the prediction model.

If there were significantly more user input data, one could build upon our model by adding that information as an input. 